In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy.io as sio
import cv2
from pathlib import Path

ROOT = Path().resolve()
if not (ROOT / 'models').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / 'data'
MODELS_DIR = ROOT / 'models'

print('ROOT:', ROOT)
print('CUDA доступна:', end=' ')
import torch; print(torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')


In [ ]:
from app.core.pipeline import Pipeline

pipeline = Pipeline()
pipeline.initialize()
print('Pipeline инициализирован')
print(f'Модель: {pipeline.detector.model_path}')


In [ ]:
detector_metrics = {
    'Precision': 0.897,
    'Recall':    0.869,
    'mAP50':     0.931,
    'F1':        2 * 0.897 * 0.869 / (0.897 + 0.869),
}

print('=== Метрики детектора (yolo11n_mot20_v2.pt) ===')
for k, v in detector_metrics.items():
    status = ''
    if k == 'Precision': status = '✅' if v >= 0.90 else '⚠️ (цель ≥90%)'
    if k == 'Recall':    status = '✅' if v >= 0.90 else '⚠️ (цель ≥90%)'
    if k == 'F1':        status = '✅' if v >= 0.88 else '⚠️ (цель ≥88%)'
    print(f'  {k}: {v:.3f} ({v*100:.1f}%) {status}')


In [ ]:
idf1_per_seq = {
    'MOT20-01': 0.719,
    'MOT20-02': 0.672,
    'MOT20-03': 0.873,
    'MOT20-05': 0.733,
}
idf1_mean = np.mean(list(idf1_per_seq.values()))

print('=== IDF1 по последовательностям MOT20 ===')
for seq, v in idf1_per_seq.items():
    print(f'  {seq}: {v*100:.1f}%')
print(f'  Среднее: {idf1_mean*100:.2f}%  ', '✅' if idf1_mean >= 0.75 else '⚠️ (цель ≥75%, −0.07%)')

print('\nПараметры ByteTrack:')
import json
params = json.load(open(ROOT / 'models' / 'bytetrack_best_params.json'))
for k, v in params.items():
    print(f'  {k}: {v}')


In [ ]:
mat = sio.loadmat(DATA_DIR / 'mall' / 'mall_dataset' / 'mall_gt.mat')
gt_counts = mat['count'].flatten().astype(int)

from app.core.detector import Detector
det_mall = Detector(conf=0.20)
det_mall.load()
from app.core.tracker import Tracker
trk_mall = Tracker()
trk_mall.init()

cap = cv2.VideoCapture(str(DATA_DIR / 'mall' / 'mall_video.mp4'))
pred_counts = []
while True:
    ret, frame = cap.read()
    if not ret: break
    dets = det_mall.detect(frame)
    tracks = trk_mall.update(dets, frame)
    pred_counts.append(len(tracks))
cap.release()

pred = np.array(pred_counts)
gt   = gt_counts[:len(pred)]
mape_mall = np.mean(np.abs(pred - gt) / np.maximum(gt, 1)) * 100
mae_mall  = np.mean(np.abs(pred - gt))

print('=== Mall Dataset — crowd counting (2000 кадров) ===')
print(f'  GT mean:   {gt.mean():.1f} чел/кадр')
print(f'  Pred mean: {pred.mean():.1f} чел/кадр')
print(f'  MAE:       {mae_mall:.2f} чел/кадр')
print(f'  MAPE:      {mape_mall:.1f}%')


In [ ]:
clips = [
    {
        'name':       'store_entrance',
        'path':       DATA_DIR / 'stock_videos' / 'uhd_30fps.mp4',
        'line_start': (0.293, 1.0),
        'line_end':   (0.293, 0.0),
        'relative':   True,
        'gt_entries': 4,
        'gt_exits':   1,
        'note':       '2560×1440, магазин, вид сбоку, вертикальная линия'
    },
]

clip_results = []

for clip in clips:
    pipeline.tracker.reset()
    pipeline.counter.reset()
    pipeline.counter.update_line(clip['line_start'], clip['line_end'],
                                  relative=clip.get('relative', False))
    result = pipeline.process_video(clip['path'])
    stats  = result['stats']

    pred_total = stats['entries'] + stats['exits']
    gt_total   = clip['gt_entries'] + clip['gt_exits']
    mape_clip  = abs(pred_total - gt_total) / max(gt_total, 1) * 100

    clip_results.append({
        'Клип':          clip['name'],
        'GT IN':         clip['gt_entries'],
        'GT OUT':        clip['gt_exits'],
        'GT total':      gt_total,
        'Pred IN':       stats['entries'],
        'Pred OUT':      stats['exits'],
        'Pred total':    pred_total,
        'MAPE (%)':      round(mape_clip, 1),
        'FPS':           result['fps'],
        'Latency мс':    result['avg_latency_ms'],
    })

df_clips = pd.DataFrame(clip_results)
print('=== Stock videos — entry/exit counting ===')
print(df_clips.to_string(index=False))

mape_clips = df_clips['MAPE (%)'].mean()
print(f'\n  Средний MAPE по {len(clips)} клипу: {mape_clips:.1f}%')


In [ ]:
perf_data = [
    {'Видео': 'MOT20-01 (test_mot20_01.mp4)', 'Разрешение': '1920x1080',
     'FPS': 82,  'Latency мс': 12.0, 'Статус': '✅'},
    {'Видео': 'Mall Dataset (640x480)',        'Разрешение': '640x480',
     'FPS': 184, 'Latency мс': 5.4,  'Статус': '✅'},
    {'Видео': 'Store entrance (UHD)',          'Разрешение': '2560x1440',
     'FPS': 239, 'Latency мс': 4.2,  'Статус': '✅'},
    {'Видео': 'Bus boarding (4K portrait)',    'Разрешение': '2160x3840',
     'FPS': 172, 'Latency мс': 5.8,  'Статус': '✅'},
]

df_perf = pd.DataFrame(perf_data)
print('=== Производительность (GPU: RTX 5080) ===')
print(df_perf.to_string(index=False))
print(f'\n  Цель latency: ≤100 мс  — все видео: ✅')
print(f'  Цель FPS: 15–30  — все видео: ✅ (значительно выше целевого)')


In [ ]:
from app.core.detector import Detector as Det2

det_fpr = Det2(conf=0.10) 
det_fpr.load()

cap = cv2.VideoCapture(str(DATA_DIR / 'mall' / 'mall_video.mp4'))
confs_all = []
for i in range(100):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i * 20)
    ret, frame = cap.read()
    if not ret: break
    dets = det_fpr.detect(frame)
    confs_all.extend([d['conf'] for d in dets])
cap.release()

confs = np.array(confs_all)
print('=== Распределение confidence детекций (Mall Dataset, conf≥0.10) ===')
for threshold in [0.10, 0.15, 0.20, 0.25, 0.30]:
    n = (confs >= threshold).sum()
    print(f'  conf ≥ {threshold:.2f}: {n} детекций из {len(confs)}')

In [ ]:
results = {
    'MAPE (≤5%)':        (None,   0.05, '<=', 'нужно 200 клипов с GT; proxy Mall: ~34%, 1 clip: 0%'),
    'Precision (≥90%)':  (0.897,  0.90, '>=', 'из notebook 02, yolo11n_mot20_v2.pt'),
    'Recall (≥90%)':     (0.869,  0.90, '>=', 'из notebook 02'),
    'F1-Score (≥88%)':   (0.881,  0.88, '>=', 'из notebook 02: 2*P*R/(P+R)'),
    'IDF1 (≥75%)':       (0.7493, 0.75, '>=', 'из notebook 03, MOT20 Extended Grid'),
    'FPR (≤3%)':         (None,   0.03, '<=', 'нет аннотированного датасета негативов'),
    'Latency (≤100мс)':  (0.0058, 0.10, '<=', 'лучший результат: 4.2 мс (UHD)'),
    'FPS (15–30)':       (172,    15,   '>=', 'все видео >>30 FPS на RTX 5080'),
}

print('=' * 80)
print(f'  {"Метрика":<22} {"Факт":>10} {"Цель":>8}  {"Статус":<8}  Примечание')
print('=' * 80)
for metric, (val, target, op, note) in results.items():
    if val is None:
        status = '⏳ нет данных'
        val_str = '—'
    else:
        if op == '<=':
            ok = val <= target
            val_str = f'{val*1000:.1f} мс' if 'Latency' in metric else f'{val*100:.1f}%' if val < 2 else str(val)
        else:
            ok = val >= target
            val_str = f'{val:.0f} FPS' if 'FPS' in metric else f'{val*100:.1f}%'
        status = '✅' if ok else '⚠️'
    target_str = f'{target*100:.0f}%' if isinstance(target, float) and target < 2 and 'FPS' not in metric and 'Latency' not in metric else str(target)
    print(f'  {metric:<22} {val_str:>10} {target_str:>8}  {status:<8}  {note}')
print('=' * 80)